In [ ]:
import os, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
  from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
  from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
from imblearn.over_sampling import SMOTE
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

# ========================
# CONFIGURACIÓN
# ========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 100
SEQ_LEN = 100 #En funcion a la cantidad de datos del esp32
LR = 1e-4
torch.manual_seed(42)
np.random.seed(42)
NUM_FOLDS = 5 # Para validación cruzada

# ========================
# CLASE PARA AUMENTO DE DATOS
# ========================
class TemporalDataAugmentation(object):
    def __init__(self, noise_level=0.01, scale_factor=0.05):
        self.noise_level = noise_level
        self.scale_factor = scale_factor

    def __call__(self, x):
        # Añadir ruido Gaussiano
        noise = torch.randn_like(x) * self.noise_level
        x = x + noise

        # Escalar los datos
        scale = 1 + torch.randn(1).item() * self.scale_factor
        x = x * scale

        return x

# ========================
# CARGA Y PREPROCESAMIENTO DE CSVs
# ========================
def load_data_from_folder(folder_path, seq_len=100):
    all_files = glob.glob(os.path.join(folder_path, "*.csv"))
    if not all_files:
        print("❌ No se encontraron archivos CSV. Asegúrate de que la ruta sea correcta.")
        return None, None
    dfs = []
    for f in tqdm(all_files, desc="📂 Cargando CSV"):
        df = pd.read_csv(f)
        df = df.drop(columns=[c for c in ["ID","TIMESTAMP"] if c in df.columns], errors="ignore")
        if "TOGGLE_VAR" in df.columns:
            df = df.rename(columns={"TOGGLE_VAR":"label"})
        for col in df.columns:
            if df[col].dtype=="object":
                df[col] = pd.to_numeric(df[col].astype(str).str.replace(",","", regex=False), errors="coerce")
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True).dropna().reset_index(drop=True)

    X = data.drop(columns=["label"]).values.astype(np.float32)
    y = data["label"].values.astype(np.int64)

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    sequences, labels = [], []
    for i in range(len(X) - seq_len):
        sequences.append(X[i:i+seq_len])
        labels.append(y[i+seq_len])
    X_seq, y_seq = np.array(sequences), np.array(labels)

    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    for train_index, test_index in sss.split(X_seq, y_seq):
        X_train, X_test = X_seq[train_index], X_seq[test_index]
        y_train, y_test = y_seq[train_index], y_seq[test_index]

    return (X_train, y_train), (X_test, y_test)

# ========================
# MODELO CNN + LSTM MEJORADO
# ========================
class CNNLSTM(nn.Module):
    def __init__(self, input_dim, cnn_channels=64, lstm_hidden=256, lstm_layers=2, dropout=0.7):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, cnn_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(cnn_channels, lstm_hidden, lstm_layers, batch_first=True, bidirectional=True)
        self.fc1 = nn.Linear(lstm_hidden*2, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.relu(self.conv1(x))
        x = x.transpose(1, 2)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        return self.fc2(out).squeeze(-1)

# ========================
# EARLY STOPPING POR F1-SCORE
# ========================
class EarlyStopF1:
    def __init__(self, patience=15, save_path="best_model.pth"):
        self.patience = patience
        self.save_path = save_path
        self.best_f1 = -1.0
        self.counter = 0
    def step(self, f1, model):
        if f1 > self.best_f1:
            self.best_f1 = f1
            self.counter = 0
            torch.save(model.state_dict(), self.save_path)
        else:
            self.counter += 1
        return self.counter >= self.patience

# ========================
# ENTRENAMIENTO
# ========================
def train(model, train_loader, valid_loader, criterion, optimizer, scheduler, epochs=100, earlystop=None):
    for epoch in range(epochs):
        model.train()
        total_loss=0
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE).float()
            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        y_true, y_logits = [], []
        val_loss=0
        with torch.no_grad():
            for X, y in valid_loader:
                X, y = X.to(DEVICE), y.to(DEVICE).float()
                logits = model(X)
                loss = criterion(logits, y)
                val_loss += loss.item()
                y_true.extend(y.cpu().numpy())
                y_logits.extend(torch.sigmoid(logits).cpu().numpy())

        y_true = np.array(y_true)
        y_logits = np.array(y_logits)

        num_pos_valid = np.sum(y_true == 1)
        if num_pos_valid > 0:
            val_pred = (y_logits >= 0.5).astype(int)
            val_recall = np.sum((y_true == 1) & (val_pred == 1)) / num_pos_valid
            val_precision = np.sum((y_true == 1) & (val_pred == 1)) / np.sum(val_pred == 1)
            val_f1 = 2 * (val_precision * val_recall) / (val_precision + val_recall)
        else:
            val_f1 = -1.0

        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {total_loss/len(train_loader):.6f} | Val F1@0.5: {val_f1:.4f}")

        if val_f1 >= 0 and earlystop is not None and earlystop.step(val_f1, model):
            print("🛑 Early stopping activado")
            break

    if earlystop is not None and os.path.exists(earlystop.save_path):
        model.load_state_dict(torch.load(earlystop.save_path, map_location=DEVICE))
    return model

# ========================
# THRESHOLD OPTIMIZADO BALANCEADO
# ========================
def optimize_threshold_balanced(y_true, y_probs, alpha=0.5):
    if len(np.unique(y_true)) < 2 or np.sum(y_true) == 0:
        return 0.5
    precision, recall, thresholds = precision_recall_curve(y_true, y_probs)
    scores = (1 + alpha**2) * (precision * recall) / ((alpha**2 * precision) + recall)
    best_idx = np.nanargmax(scores)
    best_thr = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    return best_thr

# ========================
# EVALUACIÓN
# ========================
def evaluate(model, loader, threshold=0.5):
    model.eval()
    y_true, y_logits = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(DEVICE)
            logits = model(X)
            y_logits.extend(torch.sigmoid(logits).cpu().numpy())
            y_true.extend(y.numpy())
    y_true = np.array(y_true)
    y_logits = np.array(y_logits)
    y_pred = (y_logits >= threshold).astype(int)

    print("\n=== Classification Report ===")
    print(classification_report(y_true, y_pred, digits=4, target_names=["No Caida","Caida"]))

    cm = confusion_matrix(y_true, y_pred)
    print("Confusion matrix:\n", cm)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No Caida", "Caida"], yticklabels=["No Caida", "Caida"])
    plt.title(f"Matriz de Confusión (Threshold={threshold:.2f})")
    plt.show()
    return y_true, y_logits, y_pred

# ========================
# MAIN PIPELINE MEJORADO
# ========================
if __name__=="__main__":
    (X_train_full, y_train_full), (X_test, y_test) = load_data_from_folder("/content/content", seq_len=SEQ_LEN)

    skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
    fold_results = []

    for fold, (train_index, valid_index) in enumerate(skf.split(X_train_full, y_train_full)):
        print(f"\n======== Entrenando Fold {fold+1}/{NUM_FOLDS} ========")
        X_train, X_valid = X_train_full[train_index], X_train_full[valid_index]
        y_train, y_valid = y_train_full[train_index], y_train_full[valid_index]

        X_train_flat = X_train.reshape(X_train.shape[0], -1)
        n_pos_samples = np.sum(y_train == 1)
        smote_k = 5
        if n_pos_samples > 1:
            smote_k = min(5, n_pos_samples - 1)
            sm = SMOTE(k_neighbors=smote_k, random_state=42)
            X_res, y_res = sm.fit_resample(X_train_flat, y_train)
            X_res = X_res.reshape(-1, SEQ_LEN, X_train.shape[2])
        else:
            print("⚠️ No hay suficientes muestras de la clase 'Caída' para aplicar SMOTE.")
            X_res, y_res = X_train, y_train

        # Aplicar el aumento de datos de series de tiempo
        aug = TemporalDataAugmentation()
        X_res_aug = aug(torch.from_numpy(X_res).float()).numpy()

        train_ds = TensorDataset(torch.from_numpy(X_res_aug).float(), torch.from_numpy(y_res).long())
        valid_ds = TensorDataset(torch.from_numpy(X_valid).float(), torch.from_numpy(y_valid).long())

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
        valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        model = CNNLSTM(input_dim=X_train.shape[2]).to(DEVICE)

        num_neg = np.sum(y_res == 0)
        num_pos = np.sum(y_res == 1)
        pos_weight_val = (num_neg / max(num_pos, 1)) * 0.2
        pos_weight = torch.tensor([pos_weight_val], dtype=torch.float32).to(DEVICE)
        print(f"Calculated positive weight: {pos_weight_val:.2f}")

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS) 
        earlystop = EarlyStopF1(patience=15)

        model = train(model, train_loader, valid_loader, criterion, optimizer, scheduler, epochs=EPOCHS, earlystop=earlystop)

        y_val_logits = []
        with torch.no_grad():
            for X, y in valid_loader:
                X = X.to(DEVICE)
                logits = model(X)
                y_val_logits.extend(torch.sigmoid(logits).cpu().numpy())
        y_val_logits = np.array(y_val_logits)

        best_thr = optimize_threshold_balanced(y_valid, y_val_logits, alpha=0.5)
        print(f"🔎 Mejor umbral balanceado para este fold: {best_thr:.4f}")

        fold_results.append(best_thr)

    final_best_thr = np.mean(fold_results)
    print(f"\n✅ Evaluación en test final con umbral promedio de los folds: {final_best_thr:.4f}")

    evaluate(model, DataLoader(TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test).long()), batch_size=BATCH_SIZE, shuffle=False), threshold=final_best_thr)